In [1]:
# to avoid clinical_synopsis.embedder
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

from embedder import Embedder
print("embedder import OK")

embedder import OK


# Pick 9 patients with different complexity scores

In [2]:
import pandas as pd

manifest_path = project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"

# Load manifest
df = pd.read_csv(manifest_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (50, 17)

Columns:
['filename', 'patient_id', 'patient_name', 'n_resources', 'n_encounters', 'n_observations', 'n_conditions', 'n_procedures', 'n_medication_requests', 'n_medication_administrations', 'n_diagnostic_reports', 'first_date', 'last_date', 'followup_days', 'complexity_score', 'complexity_bucket', 'sample_seed']


In [3]:
# Check bucket distribution
print("Bucket counts in full manifest:")
print(df["complexity_bucket"].value_counts())

# Sample 3 patients from each bucket (low, medium, high)
bucket_targets = {"low": 3, "medium": 3, "high": 3}
selected_rows = []

for bucket, n in bucket_targets.items():
    bucket_df = df[df["complexity_bucket"] == bucket].copy()
    if len(bucket_df) < n:
        raise ValueError(f"Not enough patients in bucket '{bucket}' to sample {n}.")

    # Random sample with a fixed seed for reproducibility
    sampled_bucket = bucket_df.sample(n=n, random_state=42)
    selected_rows.append(sampled_bucket)

selected_df = pd.concat(selected_rows).reset_index(drop=True)

print("\nSelected 9 patients (3 per bucket):")
display(selected_df[["patient_id", "patient_name", "complexity_bucket",
                     "n_resources", "complexity_score"]])

# Just the list of patient_ids for later use
selected_patient_ids = selected_df["patient_id"].tolist()
print("\nSelected patient_ids:", selected_patient_ids)

Bucket counts in full manifest:
complexity_bucket
low       17
high      17
medium    16
Name: count, dtype: int64

Selected 9 patients (3 per bucket):


,patient_id,patient_name,complexity_bucket,n_resources,complexity_score
0,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,low,230,317
1,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,low,261,330
2,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,low,436,602
3,29f6beee-162f-0113-7884-72245814693f,Eula461 Crooks415,medium,1854,2786
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,Beth967 Cremin516,medium,1843,2800
5,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,Deeann517 Torp761,medium,2191,3340
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,Francina926 Von197,high,2945,4458
7,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,Rosetta750 Stroman228,high,3055,4536
8,f203e11d-5573-1624-69b8-af8436987b3e,Shawana711 Lakin515,high,3365,4812



Selected patient_ids: ['d65197b3-056a-2136-b584-77f43c29da3f', 'f3739580-797d-ae04-eebf-aeddb2fc2f64', '4736727e-63f4-071a-1516-a49310f5a052', '29f6beee-162f-0113-7884-72245814693f', '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494', 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927', '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678', 'f203e11d-5573-1624-69b8-af8436987b3e']


In [4]:
selected_patient_ids

['d65197b3-056a-2136-b584-77f43c29da3f',
 'f3739580-797d-ae04-eebf-aeddb2fc2f64',
 '4736727e-63f4-071a-1516-a49310f5a052',
 '29f6beee-162f-0113-7884-72245814693f',
 '41681ed6-efc5-94c0-1bc0-f60b34dbd31b',
 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494',
 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 'f203e11d-5573-1624-69b8-af8436987b3e']

# chunks_df

In [5]:
# look at all oncology chunks in chunks_df:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/retrieval/metadata.db")
patient_ids = selected_patient_ids

conn = sqlite3.connect(db_path)

# Multiple patient_ids: build an IN (...) placeholder list.
if not patient_ids:
    chunks_df = pd.DataFrame()  # avoid invalid SQL: IN ()
else:
    placeholders = ",".join(["?"] * len(patient_ids))
    query = f"""
    SELECT
        chunks.patient_id,
        documents.doc_type,
        documents.title,
        chunks.heading,
        chunks.chunk_id,
        chunks.is_oncology,
        chunks.chunk_text
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id IN ({placeholders})
    """
    chunks_df = pd.read_sql_query(query, conn, params=patient_ids)

conn.close()

display(chunks_df.head())
print("Number of rows:", len(chunks_df))

onc_chunks = chunks_df[chunks_df["is_oncology"] == 1]
display(onc_chunks.head())
print("Number of oncology chunks:", len(onc_chunks))

,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
1,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,cd9bf67f542dee2c5c6eb4e889086745d239ff7b,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
2,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,c6f4af530450ec4d38f7f478693b4d62c2b3467c,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
3,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,9d5bdbca525ce28a75b1ad7deff73853cc1b20eb,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
4,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,250fb82c086a015ec8fe85d5feeb46806e632e86,0,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of rows: 14390


,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
21,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,34e083fbe248b34ab7ff75e989fc91da40a6b511,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
931,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,6c44d78bfe33c70133759592f49f88f9cd26734c,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
932,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,4f181909c2fad14ebbdc20f28f1af14d5dfb89f9,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
933,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,b02dbcb96bafeb63dc73b3c40a9f184cb0e781bf,1,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of oncology chunks: 890


In [6]:
chunks_df.columns

Index(['patient_id', 'doc_type', 'title', 'heading', 'chunk_id', 'is_oncology',
       'chunk_text'],
      dtype='str')

# f203e11d-5573-1624-69b8-af8436987b3e Shawana711 Lakin515

In [7]:
patient_id = "f203e11d-5573-1624-69b8-af8436987b3e" #high complexity patient

overview_question = "Provide a brief overview of this patient's medical background and current status."
conditions_question = "What are this patient's main diagnosed conditions and their status?"
medications_question = "What medications is this patient currently or recently taking?"
oncology_question = "Summarize this patient's oncology history."

In [8]:
import rag_service as rag

In [9]:
# # Overview 3rd version with new PATIENT_OVERVIEW_EXTRA

# overview_out = rag.rag_new(
#     query=overview_question,
#     patient_id=patient_id,
#     question_type="patient_overview",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(
#     f"=== Patient overview answer for {overview_out['patient_name']} "
#     f"(DOB {overview_out['patient_dob']}, Age: {overview_out['patient_age_years']} years, "
#     f"Gender: {overview_out['patient_gender']}) ==="
# )
# print(overview_out["answer"])

# # Conditions
# conditions_out = rag.rag_new(
#     query=conditions_question,
#     patient_id=patient_id,
#     question_type="conditions",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Conditions answer for patient {conditions_out['patient_name']} ===")
# print(conditions_out["answer"])

# # Medications
# medications_out = rag.rag_new(
#     query=medications_question,
#     patient_id=patient_id,
#     question_type="medications",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Medications answer for patient {medications_out['patient_name']} ===")
# print(medications_out["answer"])

# # Oncology timeline (if configured)
# oncology_out = rag.rag_new(
#     query=oncology_question,
#     patient_id=patient_id,
#     question_type="oncology_timeline",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Oncology timeline answer for patient {oncology_out['patient_name']} ===")
# print(oncology_out["answer"])

# ADD a question classifier

In [10]:
# # question-type classifier

# def classify_question_type(question: str) -> str:
#     q = question.lower()

#     if any(word in q for word in ["overview", "summary", "background", "history"]):
#         return "patient_overview"

#     if any(word in q for word in ["condition", "diagnosis", "diagnosed"]):
#         return "conditions"

#     if any(word in q for word in ["medication", "drug", "therapy", "prescription"]):
#         return "medications"

#     if any(word in q for word in ["oncology", "cancer", "tumor", "chemo", "radiation", "stage"]):
#         return "oncology_timeline"

#     # fallback
#     return "patient_overview"

I’m weighing the tradeoff between UX, reliability, and maintenance for your final app.

For a final app, the best default is usually not a pure keyword heuristic and not a standalone LLM classifier either. If you already have only 4 supported question types, the strongest UX is to let the user pick one when they can, and use automatic routing only as a convenience fallback.

My recommendation is:

1. If the app is a guided clinical tool, ask the user to choose one of the 4 types.
2. If the app accepts free text, use a small deterministic router first, but treat it as a helper, not as the source of truth.
3. If the router is uncertain, ask a clarification question or show the predicted type for confirmation.
4. Use an LLM classifier only if you expect lots of messy, varied, user-written questions and you can tolerate extra latency, cost, and occasional misroutes.

Why this is the best tradeoff:
- A keyword classifier like your example is fast and cheap, but it will misroute edge cases like “What happened after treatment?” or “Any evidence of progression?”.
- An LLM classifier is more flexible, but it adds cost, latency, and another failure mode, and it can still be wrong.
- Explicit user selection is the most reliable when the domain is constrained to 4 fixed categories.

For your case, I would do a hybrid:
- Show 4 buttons or a dropdown in the UI.
- If the user types a question instead, run a lightweight router.
- If the router confidence is low or the question matches multiple buckets, ask the user to choose.

So the practical answer is: use user selection as the primary path, a heuristic router as fallback, and only add an LLM classifier if you later find the routing rules are too brittle in real usage.

In [11]:
# import rag_service as rag

# # Note how whatever you ask, the answer is the same, 
# # because the question type is patient_overview, 
# # which uses the PATIENT_OVERVIEW_EXTRA prompt template.
# out = rag.rag_new(
#     query="What are the main complaints and conditions of this patient?",
#     patient_id=patient_id,
#     question_type="patient_overview",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

In [12]:
# print(
#     f"=== Patient overview answer for {overview_out['patient_name']} "
#     f"(DOB {overview_out['patient_dob']}, Age: {overview_out['patient_age_years']} years, "
#     f"Gender: {overview_out['patient_gender']}) ==="
# )
# print(overview_out["answer"])

A good heuristic router for your final app would be a small scored classifier, not a hard yes/no keyword check.

The pattern I’d use is:

- Score each of the 4 types with weighted keyword and phrase matches.
- Pick the top-scoring type only if the score is clearly above a threshold and separated from the runner-up.
- If the score is weak or ambiguous, ask the user to choose.

That gives you a simple, fast router with an uncertainty fallback.

```python
from dataclasses import dataclass

@dataclass
class RouteResult:
    question_type: str | None
    confidence: float
    scores: dict[str, float]

def route_question(question: str) -> RouteResult:
    q = question.lower()

    rules = {
        "patient_overview": [
            ("overview", 2.5),
            ("summary", 2.0),
            ("background", 1.5),
            ("history", 1.0),
            ("status", 0.5),
            ("current status", 1.5),
        ],
        "conditions": [
            ("condition", 2.5),
            ("conditions", 2.5),
            ("diagnosis", 2.0),
            ("diagnosed", 2.0),
            ("problem list", 1.5),
            ("comorbid", 1.5),
        ],
        "medications": [
            ("medication", 2.5),
            ("medications", 2.5),
            ("drug", 1.5),
            ("therapy", 1.5),
            ("prescription", 1.5),
            ("taking", 1.0),
            ("current meds", 2.0),
        ],
        "oncology_timeline": [
            ("oncology", 3.0),
            ("cancer", 2.5),
            ("tumor", 2.0),
            ("chemo", 2.5),
            ("chemotherapy", 3.0),
            ("radiation", 2.0),
            ("stage", 1.5),
            ("progression", 2.0),
            ("response", 1.5),
            ("treatment history", 2.0),
        ],
    }

    scores: dict[str, float] = {}

    for question_type, patterns in rules.items():
        score = 0.0
        for phrase, weight in patterns:
            if phrase in q:
                score += weight
        scores[question_type] = score

    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    best_type, best_score = ranked[0]
    second_score = ranked[1][1]

    # confidence heuristic: strong absolute score and clear margin
    margin = best_score - second_score
    confidence = min(1.0, best_score / 5.0)

    if best_score < 2.0 or margin < 1.0:
        return RouteResult(question_type=None, confidence=confidence, scores=scores)

    return RouteResult(question_type=best_type, confidence=confidence, scores=scores)
```

How I’d use it in the app:

```python
route = route_question(user_question)

if route.question_type is None:
    # ask user to choose from the 4 types
    ...
else:
    # call rag_new(..., question_type=route.question_type)
    ...
```

A few practical improvements make this much better than a bare keyword list:

- Add phrase weights, not just single words.
- Give oncology heavier weights because words like "history" or "summary" are too broad.
- Use both absolute threshold and margin between top two scores.
- Treat "uncertain" as a first-class output, not a failure.
- Log the routed type and confidence so you can tune rules from real usage.

For your 4-type app, this is probably the sweet spot:
- User types free text.
- Router guesses.
- If confidence is low, show four buttons: Overview, Conditions, Medications, Oncology Timeline.

I would not start with an LLM classifier unless the routing language gets much messier than this, because the heuristic router is cheaper, easier to debug, and more predictable for a clinical-style app.

In [13]:
# from dataclasses import dataclass

# @dataclass
# class RouteResult:
#     question_type: str | None
#     confidence: float
#     scores: dict[str, float]

# def route_question(question: str) -> RouteResult:
#     q = question.lower()

#     rules = {
#         "patient_overview": [
#             ("overview", 2.5),
#             ("summary", 2.0),
#             ("background", 1.5),
#             ("history", 1.0),
#             ("status", 0.5),
#             ("current status", 1.5),
#         ],
#         "conditions": [
#             ("condition", 2.5),
#             ("conditions", 2.5),
#             ("diagnosis", 2.0),
#             ("diagnosed", 2.0),
#             ("problem list", 1.5),
#             ("comorbid", 1.5),
#         ],
#         "medications": [
#             ("medication", 2.5),
#             ("medications", 2.5),
#             ("drug", 1.5),
#             ("therapy", 1.5),
#             ("prescription", 1.5),
#             ("taking", 1.0),
#             ("current meds", 2.0),
#         ],
#         "oncology_timeline": [
#             ("oncology", 3.0),
#             ("cancer", 2.5),
#             ("tumor", 2.0),
#             ("chemo", 2.5),
#             ("chemotherapy", 3.0),
#             ("radiation", 2.0),
#             ("stage", 1.5),
#             ("progression", 2.0),
#             ("response", 1.5),
#             ("treatment history", 2.0),
#         ],
#     }

#     scores: dict[str, float] = {}

#     for question_type, patterns in rules.items():
#         score = 0.0
#         for phrase, weight in patterns:
#             if phrase in q:
#                 score += weight
#         scores[question_type] = score

#     ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)
#     best_type, best_score = ranked[0]
#     second_score = ranked[1][1]

#     # confidence heuristic: strong absolute score and clear margin
#     margin = best_score - second_score
#     confidence = min(1.0, best_score / 5.0)

#     # make oncology stricter due to false positives with "history", "stage", or "progression"
#     thresholds = {
#         "patient_overview": 0.5,
#         "conditions": 0.5,
#         "medications": 0.5,
#         "oncology_timeline": 0.55,
#     }

#     if best_score < 2.0 or margin < 1.0 or confidence < thresholds[best_type]:
#         return RouteResult(question_type=None, confidence=confidence, scores=scores)

#     return RouteResult(question_type=best_type, confidence=confidence, scores=scores)

In [14]:
# # in the app

# route = route_question(user_question)

# if route.question_type is None:
#     # ask user to choose from the 4 types
#     ...
# else:
#     # call rag_new(..., question_type=route.question_type)
#     ...

In [15]:
from question_router import route_question

test_questions = [
    "Provide a brief overview of this patient's medical background and current status.",
    "What are this patient's main diagnosed conditions and their status?",
    "What medications is this patient currently or recently taking?",
    "Summarize this patient's oncology history.",
    "What happened after treatment?",
    "Any evidence of progression?",
]

for question in test_questions:
    route = route_question(question)
    print("Q:", question)
    print("Route:", route.question_type)
    print("Confidence:", round(route.confidence, 2))
    print("Scores:", route.scores)
    print("-" * 60)

Q: Provide a brief overview of this patient's medical background and current status.
Route: patient_overview
Confidence: 1.0
Scores: {'patient_overview': 6.0, 'conditions': 0.0, 'medications': 0.0, 'oncology_timeline': 0.0}
------------------------------------------------------------
Q: What are this patient's main diagnosed conditions and their status?
Route: conditions
Confidence: 1.0
Scores: {'patient_overview': 0.5, 'conditions': 7.0, 'medications': 0.0, 'oncology_timeline': 0.0}
------------------------------------------------------------
Q: What medications is this patient currently or recently taking?
Route: medications
Confidence: 1.0
Scores: {'patient_overview': 0.0, 'conditions': 0.0, 'medications': 6.0, 'oncology_timeline': 0.0}
------------------------------------------------------------
Q: Summarize this patient's oncology history.
Route: oncology_timeline
Confidence: 0.6
Scores: {'patient_overview': 1.0, 'conditions': 0.0, 'medications': 0.0, 'oncology_timeline': 3.0}
---

## Routed RAG

In [20]:
user_question = "What are the main conditions of this patient?"

In [21]:
route = route_question(user_question)

if route.question_type is None:
    print("Low confidence route.")
    print("Please choose one:")
    print("1. patient_overview")
    print("2. conditions")
    print("3. medications")
    print("4. oncology_timeline")
else:
    print("Routed to:", route.question_type)
    out = rag.rag_new(
        query=user_question,
        patient_id=patient_id,
        question_type=route.question_type,
        num_results=5,
        model="gpt-5.4-mini",
        search_type="hybrid",
    )
    print(out["answer"])

Routed to: conditions
**Diagnoses and disorders**
- Malignant neoplasm of breast (disorder) — active; date: 2015-05-07.
- Hypoxemia (disorder) — active; date: 2020-05-04.
- Viral sinusitis (disorder) — resolved; date: 2021-02-06.
- Acute pulmonary embolism (disorder) — resolved; date: 2020-05-11.
- Pneumonia (disorder) — resolved; date: 2020-05-04.
- Sepsis caused by virus (disorder) — resolved; date: 2020-05-04.
- Osteoarthritis of knee — active; date: 2004-07-12.
- Injury of medial collateral ligament of knee — resolved; date: 1979-03-02.
- Sprain of wrist — resolved; date: 2012-09-08.
- Acute bronchitis (disorder) — resolved; date: 1997-09-23.

**Findings and social/functional history**
- Stress (finding) — active; date: 2022-02-15.
- Full-time employment (finding) — active; date: 2022-02-15.
- Respiratory distress (finding) — resolved; date: 2020-05-04.
- Loss of taste (finding) — resolved; date: 2020-05-04.
- Reports of violence in the environment (finding) — resolved; date: 2019-

I’m checking the layering choice here and will answer where the router belongs in your final architecture.

I would not put the routing logic in rag_service.py.

Keep rag_service.py as the execution layer:
- search
- build context
- build prompt
- call the LLM
- return the result

Put routing in a separate small module or in the UI/app layer:
- `router.py` or `question_router.py` for the heuristic scorer
- notebook/UI code for the user-confirmation flow

Why:
- rag_service.py should stay focused on one job: answering a question once the type is known.
- Routing is a decision step before retrieval, so it belongs one layer above.
- This keeps the service reusable if later you want:
  - manual question-type selection
  - a heuristic router
  - an LLM router
  - a UI prompt for clarification

A clean flow would be:

```python
route = route_question(user_question)

if route.question_type is None:
    ask_user_to_choose()
else:
    out = rag.rag_new(
        query=user_question,
        patient_id=patient_id,
        question_type=route.question_type,
    )
```

If you want, the next good step is to factor that router into a small `clinical_synopsis/router.py` so your notebook can import it cleanly.

# LINKS

OK, I don't know how to include a link next to e.g. a value, but how about Source 1, 2, ... indication, to the documents below?



# For README

Say what the scope is and what it cannot do
Chart of Architecture? description of "Request flow"? E.g. "3. The query router identifies conversation, knowledge, pricing, schedule, or availability intent."

STACK
Language	Python 3.12+
Web interface	Streamlit
API	Flask???????

Project structure

Data source

Something like "Evaluation criteria and rubric evidence: This table maps the course rubric directly to repository evidence so reviewers do not need to infer where each requirement is implemented."



# STREAMLIT APP UI and MONITORING



1.
Add **ABOUT** section
(This application helps users understand the property purchasing process in Malaysia through a conversational interface by answering common questions in buying process, legal cost, financing, and insurance.

Answers are grounded in 16 curated Malaysian property guides.)

2.
📖 **How to Use**

Click 🔑 API Settings in the sidebar to use your own API key and model.
Type a question in the chat box below.
The assistant retrieves relevant guides and generates an answer.
Click Sources to view the original articles.
Upvote 👍 or downvote 👎 each answer to help us improve our quality.
**Example** questions:
How much money do I need upfront?
What documentations needed for the purchase?
Are there any government schemes to help me buy my first home?
Is there any special incentives for first time buyer?
What type of insurance available and which mandatory?
What legal fees should I prepare when buying a house?
How to calculate how much financing I can obtain?
What Islamic financing options available?


3. MONITORING in the sidebar


Total Queries
2
Avg Latency
14044 ms
Total Cost
$0.0004
Avg Tokens
786
Relevance Rate
50%


CHARTS
- Overview:
Query Volume
Cost Over Time
Token Usage
User Feedback
- Performance:
Latency Distribution
Latency Trend
Cost Distribution
Top Cited Sources
- Models =====> BECAUSE the app allows you to give your own API key
Query Count by Model
Total Cost by Model ($)
- Judge: =====> !!!!
Relevance Distribution
Relevance Rate Trend (%)
Latest Judge Verdicts:
What is the price range for 1 bed apartments?
RELEVANT
The answer directly provides a price range for 1‑bed apartments, which is exactly what the question asks.
🕒 Aug 13, 15:09
🤖 openai/gpt-oss-120b
⚡ 1429 ms
What are the cheapest areas to buy a studio apartment?
PARTLY_RELEVANT
The answer acknowledges the question about cheapest areas for a studio apartment but states that the provided sources do not contain that information, so it cannot give a specific answer. It addresses the query but fails to supply the requested details, making it only partially relevant.












# Dashboard Metrics

Yes. Your existing feedback is still useful for several strong dashboard metrics because it already stores timestamps, ratings, issue flags, question type, retrieval method, model, token counts, and estimated per-request cost.

The main thing it does **not** currently store is latency or a judge score. Those need to be captured during answer generation and saved with each feedback event.

## Best charts to add

For a Zoomcamp project, I would prioritize these:

| Dashboard item | What it answers | Existing data sufficient? |
|---|---|---|
| **Total queries** | How many generated answers were reviewed/saved in total? | Yes |
| **Query volume over time** | How many queries occurred per day or week? | Yes, uses `created_at` |
| **Total estimated cost** | What is the cumulative cost of all tracked requests? | Yes, uses `total_cost` |
| **Cost over time** | Is cumulative spend rising over days/weeks? | Yes |
| **Cost by retrieval method** | Does hybrid, semantic, or lexical retrieval cost more? | Yes |
| **Cost by question type** | Which clinical question categories cost more? | Yes |
| **Input/output token trends** | Are prompts or answers getting longer? | Yes |
| **Tokens by question type** | Which kinds of questions consume the most context/output? | Yes |
| **Feedback score over time** | Is answer quality improving or deteriorating? | Yes |
| **Accuracy flags over time** | Are clinicians flagging more potential inaccuracies? | Yes |
| **Routing distribution** | Which question types does the router select most often? | Yes |
| **Latency distribution** | How long do users wait for an answer? | No—add `latency_seconds` |
| **Judge score over time** | Does automated relevance agree with clinician feedback? | No—add judge fields |

### Total queries vs. query volume

They are related but not the same:

- **Total queries** is one number: for example, “148 queries have been generated.”
- **Query volume** is a time-series chart: for example, “12 queries Monday, 23 Tuesday, 9 Wednesday.”

For the dashboard, include both:

```python
st.metric("Total queries", f"{len(filtered):,}")
```

And a daily-volume chart:

```python
daily_queries = (
    filtered
    .dropna(subset=["created_at"])
    .assign(date=lambda df: df["created_at"].dt.date)
    .groupby("date")
    .size()
    .rename("queries")
)

st.line_chart(daily_queries)
```

A single metric gives a quick scale indicator; a daily chart shows whether usage is growing, spiking, or declining. Streamlit supports metric cards as well as dataframe-backed charts. [docs.streamlit](https://docs.streamlit.io/develop/api-reference/data/st.metric)

## Useful additions now

Add these after your current “Quality summary” section.

```python
st.subheader("Usage and cost")

usage_columns = st.columns(4)

usage_columns[0].metric(
    "Total queries",
    f"{len(filtered):,}",
)

usage_columns [docs.streamlit](https://docs.streamlit.io/develop/api-reference/data/st.metric).metric(
    "Total input tokens",
    f"{filtered['input_tokens'].fillna(0).sum():,.0f}",
)

usage_columns [github](https://github.com/streamlit/docs/blob/main/content/develop/quick-references/api-cheat-sheet.md).metric(
    "Total output tokens",
    f"{filtered['output_tokens'].fillna(0).sum():,.0f}",
)

usage_columns [developers.openai](https://developers.openai.com/api/docs/guides/advanced-usage).metric(
    "Total estimated cost",
    f"${filtered['total_cost'].fillna(0).sum():.4f}",
)
```

Then add three compact charts:

```python
chart_left, chart_right = st.columns(2)

with chart_left:
    st.subheader("Query volume by day")

    daily_queries = (
        filtered
        .dropna(subset=["created_at"])
        .assign(date=lambda df: df["created_at"].dt.date)
        .groupby("date")
        .size()
        .rename("queries")
    )

    if daily_queries.empty:
        st.caption("No dated query records are available.")
    else:
        st.line_chart(daily_queries)

with chart_right:
    st.subheader("Estimated cost by question type")

    cost_by_question_type = (
        filtered
        .groupby("question_type")["total_cost"]
        .sum()
        .sort_values(ascending=False)
        .rename("estimated_cost")
    )

    if cost_by_question_type.empty:
        st.caption("No cost data is available.")
    else:
        st.bar_chart(cost_by_question_type)
```

And a token breakdown:

```python
st.subheader("Token usage by retrieval method")

tokens_by_search_type = (
    filtered
    .groupby("search_type")[["input_tokens", "output_tokens"]]
    .sum()
    .fillna(0)
)

if tokens_by_search_type.empty:
    st.caption("No token-usage data is available.")
else:
    st.bar_chart(tokens_by_search_type)
```

Input and output tokens are useful because they influence both cost and response behavior; API usage metadata commonly exposes token counts through a response’s usage field. [developers.openai](https://developers.openai.com/api/docs/guides/advanced-usage)

## Add latency tracking

Latency is worthwhile, but only after you measure it at generation time. Store the full elapsed time for `rag_new()`—from request start until an answer is ready.

### In `clinical_synopsis_app.py`

Add this import:

```python
import time
```

Then change your current generation block from:

```python
with st.spinner("Retrieving the patient record and generating an answer..."):
    result = rag_new(
        query=question.strip(),
        patient_id=patient_id,
        question_type=question_type,
        search_type=search_type,
        model=model,
    )
```

to:

```python
with st.spinner("Retrieving the patient record and generating an answer..."):
    started_at = time.perf_counter()

    result = rag_new(
        query=question.strip(),
        patient_id=patient_id,
        question_type=question_type,
        search_type=search_type,
        model=model,
    )

    latency_seconds = time.perf_counter() - started_at
```

Store it in `st.session_state.response`:

```python
"latency_seconds": latency_seconds,
```

And save it in `feedback_event`:

```python
"latency_seconds": response["latency_seconds"],
```

### In `feedback.py`

Add this database column to the table definition:

```sql
latency_seconds REAL,
```

And add it to both the `INSERT INTO feedback (...)` column list and `VALUES (...)` list:

```python
latency_seconds,
```

```python
:latency_seconds,
```

For an already-created database, add a lightweight migration:

```python
try:
    conn.execute(
        "ALTER TABLE feedback ADD COLUMN latency_seconds REAL"
    )
except sqlite3.OperationalError:
    pass
```

Place it immediately after `CREATE TABLE IF NOT EXISTS feedback (...)`.

Then add a latency chart:

```python
st.subheader("Answer-generation latency")

latency_by_type = (
    filtered
    .dropna(subset=["latency_seconds"])
    .groupby("question_type")["latency_seconds"]
    .mean()
    .sort_values(ascending=False)
    .rename("average_seconds")
)

if latency_by_type.empty:
    st.caption("Latency will appear after new answer requests are recorded.")
else:
    st.bar_chart(latency_by_type)
```

Use **median** and **P95** latency once you have more records. Averages can hide occasional very slow requests, and percentiles are more useful for understanding user-facing delay. [help.openai](https://help.openai.com/en/articles/1000499-troubleshooting-api-errors-and-latency)

## Judge integration

Yes—adding your `evaluate_relevance()` judge is probably easy **if it already accepts the generated answer and retrieved context/documents**.

The best approach is to run it immediately after `rag_new()` creates the answer, not later inside the monitoring dashboard. The dashboard should **display stored evaluation results**, rather than recompute them each time someone opens a page.

Conceptually:

```python
result = rag_new(...)

judge_result = evaluate_relevance(
    question=question.strip(),
    answer=result["answer"],
    retrieved_context=result["context"],
)
```

Then store the judge output alongside the answer:

```python
"judge_score": judge_result["score"],
"judge_reason": judge_result["reason"],
"judge_label": judge_result["label"],
```

For example, your judge might return:

```python
{
    "score": 4,
    "label": "relevant",
    "reason": "The answer addresses the medication question and is supported by retrieved records."
}
```

## Judge fields to save

Add these fields to your SQLite `feedback` table:

```sql
judge_score REAL,
judge_label TEXT,
judge_reason TEXT,
```

Then save them with each event:

```python
"judge_score": response["judge_score"],
"judge_label": response["judge_label"],
"judge_reason": response["judge_reason"],
```

This enables useful comparison charts:

- Average judge score by retrieval method.
- Average judge score by question type.
- Judge-score trend over time.
- Clinician score versus judge score.
- Low clinician rating but high judge score: likely a judge blind spot.
- High clinician rating but low judge score: perhaps the judge is too strict.
- Accuracy-flag rate by judge label.

## Recommended final dashboard

For the project, I would keep it to five logical sections:

1. **Usage:** total queries, query volume by day, queries by question type.
2. **Cost and tokens:** total cost, cost trend, token usage by retrieval method.
3. **Quality:** average clinician score, rating distribution, accuracy flags, issue categories.
4. **Performance:** median latency, P95 latency, latency by retrieval method.
5. **Evaluation:** average judge score, judge-score distribution, judge-versus-clinician comparison.

Start with the metrics supported by your existing database. Add latency and judge-score persistence next. That creates a coherent observability story: **how much the system is used, what it costs, how it performs, and whether humans and automated evaluation consider its answers useful.**

### MY NOTES

section order is good: load → filter → summary → charts → table → answer inspection

Your final dashboard order should be:

Quality summary
- Quality metrics
- Rating distribution
- Issue categories

Usage and cost
- Feedback submissions
- Token totals
- Total estimated cost
- Feedback volume by day
- Cost by question type
- Token usage by retrieval method

Performance
- Latency chart

Feedback records
- Interactive data table
- CSV download

Inspect an answer
- Question, answer, comment, metadata, source preview


Your page’s current structure is already suitable for a separate Automated evaluation section beneath latency.